# Factor Spanning Tests

A profitable factor is not necessarily a useful addition to a portfolio. Its returns may reflect exposures the investor already holds. To assess its contribution, I ask whether it earns an average return beyond what those existing exposures explain.

Factor-spanning tests address this question with time-series regressions. In this notebook, I first explain the benchmark, the interpretation of alpha, and the distinction between return exposure and investment value. I then use the published Fama–French factor returns to ask whether size, value, profitability, investment, and momentum strategies add to a market-only benchmark. Notebook 08 applies factor models to test portfolios and evaluates joint pricing restrictions.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
from factor_asset_pricing.french_data import load_french, sample_coverage
RAW = ROOT / "data/raw/french"
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from factor_asset_pricing.asset_pricing_tests import factor_regressions
pd.options.display.float_format="{:,.3f}".format
plt.style.use("seaborn-v0_8-whitegrid")
factors = load_french("ff5", RAW).loc["1963-07":"2024-12"]
assert len(factors) == 738 and not factors.isna().any().any()
momentum = load_french("momentum", RAW)["MOM"].reindex(factors.index)
assert momentum.notna().all()
factor_panel = factors[["MKT", "SMB", "HML", "RMW", "CMA"]].join(momentum)
assert len(factor_panel) == 738 and not factor_panel.isna().any().any()
display(sample_coverage(factor_panel))


## 1. Factor spanning tests and time-series regressions

### 1.1 Asset-pricing benchmarks

When considering an additional factor, we want to know whether it offers expected return beyond what its exposure to the existing factors explains. A factor-spanning test evaluates this contribution. The existing factors form our benchmark, which can contain one factor or several.

Alpha depends on what we compare against. A factor might add something beyond the market, but little once we include other factors that explain its average return. A significant loading tells us that the factor has exposure to a benchmark factor. It does not, on its own, tell us that this exposure earns compensation. We therefore need to choose the benchmark before interpreting alpha.


### 1.2 Alpha, exposure, and spanning

For factor spanning tests, we put the candidate factor's excess-return payoff on the left-hand side and the benchmark factors on the right:

$$F_{c,t}=\alpha+\boldsymbol\beta'\mathbf F_t+\varepsilon_t.$$

Here $F_{c,t}$ is the candidate factor payoff and $\mathbf F_t$ contains the benchmark factor payoffs. The intercept $\alpha$ captures the average return left unexplained by those exposures, $\boldsymbol\beta$ contains the risk loadings, and $\varepsilon_t$ is the regression disturbance. In the factor spanning test, the factors must be expressed as excess returns. However, remember from notebook 03 that long–short factors already are excess-return payoffs. We therefore do not subtract the risk-free return again. The benchmark factors also enter as excess or self-financing returns.

In the population linear projection, $E[\varepsilon_t]=0$ and $E[\varepsilon_t\mathbf F_t]=\mathbf0$. Taking expectations gives

$$\alpha=E[F_{c,t}]-\boldsymbol\beta'E[\mathbf F_t].$$

This shows us that alpha measures the average return left unexplained by the asset-pricing benchmark. In addition, we use adjusted $R^2$ in empirical asset pricing. This tells us how much of the monthly return variation in the left-hand-side factor the model explains, accounting for the number of regressors. A model can explain much of the monthly variation and still miss the average return. Equally, low fit alone does not show that adding the factor improves our investment opportunities.

### 1.3 Estimation and inference

#### Hypothesis tests
To test whether the candidate adds expected return beyond its benchmark exposures, we test

$$H_0:\alpha=0,\qquad H_1:\alpha\neq0.$$

Here we use one set of population regression coefficients rather than allowing exposures or expected returns to change with the information available at each date. This is an unconditional comparison.

For each exposure $\beta_k$, we also test

$$H_0:\beta_k=0,\qquad H_1:\beta_k\neq0.$$

A significantly positive loading means that the candidate tends to move with benchmark factor $k$, holding the other benchmark factors fixed. A significantly negative loading means that it tends to move in the opposite direction. This tells us about exposure, rather than whether the candidate earns alpha.

#### Statistical Inference

We estimate the relationship between the candidate payoff and its benchmark using OLS. In the formula below, $X$ contains a constant and the benchmark factor observations, while $\mathbf y$ contains the candidate factor payoffs. The OLS coefficient estimates are

$$\widehat{\boldsymbol\theta}=(X'X)^{-1}X'\mathbf y,\qquad\boldsymbol\theta=(\alpha,\boldsymbol\beta')'.$$

For a coefficient $\theta$ and its hypothesized value $\theta_0$, we compute the t-statistic

$$t_{NW}=\frac{\widehat\theta-\theta_0}{SE_{NW}(\widehat\theta)}.$$

For alpha, $\theta_0=0$. The statistic tells us how far the estimated alpha is from zero, measured in standard errors. Since financial data often exhibit heteroskedasticity and autocorrelation in the errors, empirical asset-pricing researchers commonly use Newey–West standard errors. Note that this inference relies on an asymptotic approximation.

#### Newey–West covariance estimation

The Bartlett-weighted Newey–West estimator of the long-run covariance of the regression scores is

$$\widehat S=\sum_{j=-k}^{k}\left(\frac{k-|j|}{k}\right)\frac1T\sum_{\substack{1\le t\le T\\1\le t-j\le T}}\mathbf u_t\mathbf u_{t-j}'.$$

Here $T$ is the sample size, $k$ is the bandwidth, and $j$ indexes the lags. The vector $\mathbf x_t$ contains the constant and benchmark factor observations, and $\widehat\varepsilon_t$ is the OLS residual. Then $\mathbf u_t=\mathbf x_t\widehat\varepsilon_t$ is the regression score. The inner sum uses pairs observed within the sample. The Bartlett weight $(k-|j|)/k$ gives less weight to longer lags.

In this notation, the weight at lag $k$ is zero. The code includes $L=12$ monthly lags with weights $1-j/(L+1)$, so the corresponding bandwidth here is $k=L+1=13$. The notation differs, but the weights are the same. These are the standard errors that I use throughout when performing my asset pricing tests. The library uses this estimate to calculate coefficient standard errors. The OLS coefficients and fitted returns remain unchanged.


## 2. Example: do the characteristic and momentum factors add to the market?

### 2.1 Data and benchmark

I use published monthly returns from the [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), with a sample that starts in July 1963 and, in my case, ends in December 2024.

For my factor spanning test, I use the market as the benchmark and treat SMB, HML, RMW, CMA, and MOM as candidate investments. That is, I test whether each provides abnormal returns beyond the market. MOM is the published cross-sectional momentum factor introduced in [04](04_momentum_in_asset_pricing.ipynb), where I use the label CSMOM to distinguish it from time-series momentum. Here I retain the source label MOM. Like the other long–short factors, it already represents a self-financing payoff, so I do not subtract RF again. All five comparisons use the same months. Theoretically, under CAPM, market exposure should explain expected excess returns, leaving zero alpha. Hence, this simple example also provides direct insight into the empirical failure of CAPM and why investors, both in industry and in academia, are now looking for new factors to find new sources of alpha.

In [ ]:
anomaly_factors=factor_panel[["SMB","HML","RMW","CMA","MOM"]]
capm=factor_regressions(anomaly_factors,factor_panel[["MKT"]],cov_type="HAC",hac_lags=12)
capm_ols=factor_regressions(anomaly_factors,factor_panel[["MKT"]],cov_type="OLS")

### 2.2 CAPM spanning results

The table below provides the results from the spanning test and compares conventional OLS standard errors and Newey–West adjusted standard errors to show the difference. Alpha and both standard errors are in percent per month. Market beta, coefficient statistics, and adjusted $R^2$ are dimensionless. The OLS statistic conventionally uses a finite-sample Student distribution under the classical normal-error assumptions, whereas the Newey–West statistic uses the asymptotic approximation explained above.

In [ ]:
results=pd.DataFrame({
    "Alpha (%)":capm["alpha"]*100,
    "OLS SE (%)":capm_ols["alpha_se"]*100,
    "OLS t":capm_ols["alpha_t"],
    "NW SE (%)":capm["alpha_se"]*100,
    "NW t":capm["alpha_t"],
    "Market beta":capm["beta_MKT"],
    "NW beta t":capm["beta_MKT_t"],
    "Adjusted R²":capm["adjusted_r_squared"],
})
display(results)

The first thing that is worth noticing is that Newey–West increases the alpha standard error for SMB, HML, RMW, and CMA in this sample, while reducing it for MOM. The adjustment accounts for autocorrelation and heteroskedasticity in the errors, but it need not increase the standard error.

When looking at the alphas more directly, HML, RMW, and CMA have monthly CAPM alphas of approximately 0.361%, 0.338%, and 0.346%. Their absolute Newey–West $t$-statistics exceed 1.96, so each rejects zero alpha at the nominal 5% significance level. Market exposure therefore does not explain the average returns of these Fama–French factors in this sample, which is expected.

Furthermore, SMB has a positive market loading, while HML, RMW, CMA, and MOM have negative loadings. This means that when the market increases, SMB tends to increase as well, whereas the other factors tend to decrease. The Newey–West loading statistics are approximately 6.572 for SMB, −2.599 for HML, −2.293 for RMW, and −5.397 for CMA. MOM has a Newey–West loading statistic of −2.074. All five loadings are statistically significant at the nominal 5% level. It is important to remember that this is in-sample behavior and may not hold out of sample, for example, if one decides to invest in the factors after 2024.

For an investor already holding market exposure, the positive alphas suggest that HML, RMW, CMA, and MOM may improve the available investment opportunities. So a person who wishes to invest in a portfolio that offers expected return beyond what market exposure explains may be interested in investing in these Fama–French factors. The in-sample results do not guarantee that the additional return will persist.

MOM also retains a positive CAPM alpha: approximately 0.703% per month, with a Newey–West $t$-statistic of 5.145. Its negative market loading and low adjusted $R^2$ indicate that market exposure captures little of its monthly variation. The momentum payoff therefore adds average return beyond this benchmark in the sample, without identifying the economic forces behind momentum.

### 2.3 Visualizing the regressions

I plot each factor's monthly payoff against the market excess return to make the fitted relationship easier to see. The intercept is the estimated alpha, and the slope is the market beta.


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
x=factor_panel["MKT"]*100
for ax,name in zip(axes.flat,anomaly_factors):
    sample=pd.concat([x.rename("MKT"),(anomaly_factors[name]*100).rename(name)],axis=1).dropna()
    grid=np.linspace(sample.MKT.min(),sample.MKT.max(),100)
    alpha=capm.loc[name,"alpha"]*100
    beta=capm.loc[name,"beta_MKT"]
    ax.scatter(sample.MKT,sample[name],s=9,alpha=.22,color="#4C78A8",linewidths=0)
    ax.plot(grid,alpha+beta*grid,color="#B24A35",lw=1.5)
    ax.set(title=f"{name}: α={alpha:.2f}% (t={capm.loc[name,'alpha_t']:.2f}), β={beta:.2f}",xlabel="MKT return (%)",ylabel=f"{name} return (%)")
for ax in axes.flat[len(anomaly_factors.columns):]:
    ax.set_visible(False)
fig.suptitle("CAPM spanning regressions",weight="bold")
plt.show()


The plots tell the same story as the table. The fitted lines have shallow slopes, and observations are widely scattered around them, consistent with the low adjusted $R^2$ values. HML, RMW, CMA, and MOM have positive intercepts, while SMB's intercept is too imprecisely estimated to distinguish from zero.

I use the plots to understand the relationship, not to judge statistical significance. For that, I use the Newey–West $t$-statistics in the table.


## References

- Kenneth R. French Data Library. [Five-factor data and construction](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).

- Kenneth R. French Data Library. [Monthly momentum factor construction](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_mom_factor.html).
